## Handle Data

#### Corta mosaico que será inferido em imagens de tamanho fixo para serem utilizadas como entrada da rede

In [1]:
import os
import rasterio
from tqdm import tqdm
from rasterio.windows import Window

def process_single_image(raster_path, output_dir, tile_size=512):
    """Processa uma única imagem gerando tiles completos."""
    with rasterio.open(raster_path) as raster_src:
        # Determina o número de tiles completos em cada dimensão
        num_rows = raster_src.height // tile_size
        num_cols = raster_src.width // tile_size
        
        if num_rows == 0 or num_cols == 0:
            print(f"Imagem {os.path.basename(raster_path)} muito pequena para tiles de {tile_size}x{tile_size}")
            return 0

        base_name = os.path.splitext(os.path.basename(raster_path))[0]
        tile_count = 0

        for row in range(num_rows):
            for col in range(num_cols):
                # Calcula a posição do tile
                y = row * tile_size
                x = col * tile_size
                window = Window(x, y, tile_size, tile_size)

                try:
                    # Lê o tile
                    raster_tile = raster_src.read(window=window)
                    
                    # Verifica tamanho exato
                    if raster_tile.shape[1:] != (tile_size, tile_size):
                        continue
                    
                    
                    # Incrementa contador e salva
                    tile_count += 1
                    
                    # Salva imagem
                    output_path = os.path.join(output_dir, f"{base_name}_tile_{tile_count}.tif")
                    with rasterio.open(output_path, 'w',
                                    driver='GTiff',
                                    height=tile_size,
                                    width=tile_size,
                                    count=raster_src.count,
                                    dtype=raster_tile.dtype,
                                    crs=raster_src.crs,
                                    transform=rasterio.windows.transform(window, raster_src.transform)) as dst:
                        dst.write(raster_tile)
                
                except Exception as e:
                    print(f"Erro no tile {row},{col}: {str(e)}")
                    continue
        
        return tile_count

# Configurações
RASTER_DIR = 'MOSAIC/'
OUTPUT_X_DIR = 'CROPPED_MOSAIC/1999/'
TILE_SIZE = 512

# Certifique-se que o diretório de saída existe
os.makedirs(OUTPUT_X_DIR, exist_ok=True)

# Lista todos os arquivos no diretório RASTER_DIR (ajuste os formatos conforme necessário)
supported_formats = ['.tif', '.tiff', '.geotiff', '.jpg', '.png']
image_files = [f for f in os.listdir(RASTER_DIR) 
              if os.path.splitext(f)[1].lower() in supported_formats]

# Processa cada imagem
total_tiles = 0
for img_file in tqdm(image_files, desc="Processando imagens"):
    img_path = os.path.join(RASTER_DIR, img_file)
    tiles_generated = process_single_image(img_path, OUTPUT_X_DIR, TILE_SIZE)
    total_tiles += tiles_generated
    print(f"{img_file}: {tiles_generated} tiles gerados")

print(f"\nProcessamento concluído! Total de tiles gerados: {total_tiles}")

Processando imagens: 100%|██████████| 1/1 [11:34<00:00, 694.49s/it]

MOSAIC_1999.tif: 17316 tiles gerados

Processamento concluído! Total de tiles gerados: 17316


#### Descarta imagens totalmente pretas

In [2]:
import os
import rasterio
from tqdm import tqdm
import numpy as np

def is_black_image(image_path, threshold=0):
    """Verifica se a imagem é completamente preta (ou abaixo do threshold)."""
    try:
        with rasterio.open(image_path) as src:
            # Lê todos os pixels da imagem
            data = src.read()
            
            # Verifica se todos os pixels em todas as bandas são <= threshold
            if len(data.shape) == 3:
                return np.all(data <= threshold)
            else:
                return np.all(data <= threshold)
    except Exception as e:
        print(f"Erro ao processar {image_path}: {str(e)}")
        return False

def delete_black_images(tiles_dir, threshold=0):
    """Deleta imagens completamente pretas no diretório especificado."""
    # Lista todos os arquivos no diretório (ajuste os formatos conforme necessário)
    supported_formats = ['.tif', '.tiff', '.geotiff', '.jpg', '.png']
    image_files = [f for f in os.listdir(tiles_dir) 
                 if os.path.splitext(f)[1].lower() in supported_formats]
    
    deleted_count = 0
    for img_file in tqdm(image_files, desc="Verificando imagens pretas"):
        img_path = os.path.join(tiles_dir, img_file)
        if is_black_image(img_path, threshold):
            try:
                os.remove(img_path)
                deleted_count += 1
            except Exception as e:
                print(f"Erro ao deletar {img_path}: {str(e)}")
    
    print(f"\nProcessamento concluído! Total de imagens pretas deletadas: {deleted_count}")

# Configurações
TILES_DIR = 'CROPPED_MOSAIC/1999/'
THRESHOLD = 0  # Ajuste este valor conforme necessário

# Executa a limpeza
delete_black_images(TILES_DIR, THRESHOLD)

Verificando imagens pretas: 100%|██████████| 17316/17316 [13:16<00:00, 21.74it/s] 


Processamento concluído! Total de imagens pretas deletadas: 7793


## Inferência

In [4]:
import os
import torch
import rasterio
from torchvision import transforms
import numpy as np
import torchvision

# Configurações
X_DIR = 'CROPPED_MOSAIC/1999/'  # Pasta com imagens de entrada
OUTPUT_DIR = 'INFERENCIA/'  # Pasta para salvar as predições em grayscale
MODEL_PATH = 'best_model_1999.pth'  # Caminho para o melhor modelo salvo
NUM_CLASSES = 3  # Número de classes no seu modelo

# Criar pasta de saída
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Dispositivo (GPU se disponível)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Carregar modelo
def create_deeplabv3(output_channels=3):
    weights = torchvision.models.segmentation.DeepLabV3_ResNet50_Weights.DEFAULT
    model = torchvision.models.segmentation.deeplabv3_resnet50(weights=weights)
    model.classifier[4] = torch.nn.Conv2d(256, output_channels, kernel_size=(1, 1))
    if model.aux_classifier is not None:
        model.aux_classifier[4] = torch.nn.Conv2d(256, output_channels, kernel_size=(1, 1))
    return model

model = create_deeplabv3(output_channels=NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

# Transformações
transform = transforms.Compose([
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Processar cada imagem
for filename in os.listdir(X_DIR):
    if filename.endswith('.tif'):
        input_path = os.path.join(X_DIR, filename)
        output_path = os.path.join(OUTPUT_DIR, filename)
        
        print(f"Processando: {filename}")
        
        # Carregar imagem
        with rasterio.open(input_path) as src:
            x = src.read()[:3]  # Pega apenas os 3 primeiros canais (RGB)
            meta = src.meta.copy()
        
        # Atualizar metadados para imagem de saída (1 banda, grayscale)
        meta.update({
            'count': 1,
            'dtype': 'uint8'
        })
        
        # Pré-processamento
        x = torch.from_numpy(x).float() / 255.0
        x = transform(x)
        x = x.unsqueeze(0).to(device)
        
        # Inferência
        with torch.no_grad():
            output = model(x)['out']
            pred = torch.argmax(output, dim=1).squeeze().cpu().numpy()
        
        # Converter predições para uint8 (0-255)
        pred_gray = pred.astype(np.uint8)
        
        # Salvar TIFF grayscale georreferenciado
        with rasterio.open(output_path, 'w', **meta) as dst:
            dst.write(pred_gray, 1)  # Escreve na banda 1

print("Inferência concluída! Arquivos em grayscale salvos em:", OUTPUT_DIR)

Usando dispositivo: cuda
Processando: MOSAIC_1999_tile_10009.tif
Processando: MOSAIC_1999_tile_10010.tif
Processando: MOSAIC_1999_tile_10011.tif
Processando: MOSAIC_1999_tile_10012.tif
Processando: MOSAIC_1999_tile_10013.tif
Processando: MOSAIC_1999_tile_10014.tif
Processando: MOSAIC_1999_tile_10015.tif
Processando: MOSAIC_1999_tile_10016.tif
Processando: MOSAIC_1999_tile_10017.tif
Processando: MOSAIC_1999_tile_10018.tif
Processando: MOSAIC_1999_tile_10019.tif
Processando: MOSAIC_1999_tile_10020.tif
Processando: MOSAIC_1999_tile_10021.tif
Processando: MOSAIC_1999_tile_10022.tif
Processando: MOSAIC_1999_tile_10023.tif
Processando: MOSAIC_1999_tile_10024.tif
Processando: MOSAIC_1999_tile_10025.tif
Processando: MOSAIC_1999_tile_10026.tif
Processando: MOSAIC_1999_tile_10027.tif
Processando: MOSAIC_1999_tile_10028.tif
Processando: MOSAIC_1999_tile_10029.tif
Processando: MOSAIC_1999_tile_10030.tif
Processando: MOSAIC_1999_tile_10031.tif
Processando: MOSAIC_1999_tile_10032.tif
Processando: MO

In [5]:
import os
import rasterio
from rasterio.merge import merge

# --- Configurações ---
# Pasta onde estão as imagens de inferência (grayscale TIFFs)
INPUT_DIR = 'INFERENCIA/' 

# Pasta e nome do arquivo de saída para o mosaico
OUTPUT_DIR = 'MOSAICO/'
OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'mosaico_final.tif')

# --- Início do Script ---

# 1. Criar a pasta de saída se ela não existir
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Encontrar todos os arquivos .tif na pasta de entrada
try:
    # Lista para armazenar os caminhos dos arquivos a serem unidos
    src_files_to_mosaic = []
    for filename in os.listdir(INPUT_DIR):
        if filename.endswith('.tif'):
            full_path = os.path.join(INPUT_DIR, filename)
            src_files_to_mosaic.append(full_path)

    if not src_files_to_mosaic:
        print(f"Nenhum arquivo .tif foi encontrado na pasta '{INPUT_DIR}'.")
    else:
        print(f"Encontrados {len(src_files_to_mosaic)} arquivos para criar o mosaico.")

        # 3. Abrir todos os arquivos TIFF que serão unidos
        src_datasets = [rasterio.open(fp) for fp in src_files_to_mosaic]

        # 4. Unir (merge) os arquivos em um único mosaico
        # A função merge cuida do alinhamento espacial baseado nos metadados
        print("Iniciando a criação do mosaico...")
        mosaic, out_trans = merge(src_datasets)
        
        # 5. Copiar os metadados de uma das imagens e atualizá-los para o novo mosaico
        out_meta = src_datasets[0].meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_trans,
            "crs": src_datasets[0].crs  # Garante que o sistema de coordenadas seja mantido
        })

        # 6. Salvar o mosaico em um novo arquivo .tif
        print(f"Salvando o mosaico em: {OUTPUT_PATH}")
        with rasterio.open(OUTPUT_PATH, "w", **out_meta) as dest:
            dest.write(mosaic)

        # 7. Fechar todos os datasets abertos
        for src in src_datasets:
            src.close()

        print("Criação do mosaico concluída com sucesso!")

except Exception as e:
    print(f"Ocorreu um erro: {e}")



Encontrados 9523 arquivos para criar o mosaico.
Iniciando a criação do mosaico...
Salvando o mosaico em: MOSAICO/mosaico_final.tif
Criação do mosaico concluída com sucesso!
